In [ ]:
!pip uninstall -y openai-agents
%pip install --no-cache-dir openai-agents==0.2.2
%pip install python-dotenv


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.1/161.1 kB 38.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 150.7/150.7 kB 149.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 234.6/234.6 kB 251.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 948.6/948.6 kB 91.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.4/80.4 kB 225.7 MB/s eta 0:00:00
  Attempting uninstall: openai
    Found existing installation: openai 2.54.0
    Uninstalling openai-2.54.0:
      Successfully uninstalled openai-2.54.0


In [ ]:
import os
from agents import AsyncOpenAI, set_tracing_disabled
from IPython.display import display, Markdown

def get_gemini_key():
    # Option A: Google Colab Secrets (key icon in the left sidebar)
    try:
        from google.colab import userdata
        for name in ("GEMINI_API_KEY", "GOOGLE_API_KEY"):
            try:
                key = userdata.get(name)
                if key:
                    print(f"Key loaded from Colab Secrets ({name})")
                    return key
            except Exception:
                pass   # secret missing or notebook access switched off
    except ImportError:
        pass           # not running in Colab
    # Option B: .env file on your laptop
    try:
        from dotenv import load_dotenv
        load_dotenv()
    except ImportError:
        pass
    key = os.getenv("GEMINI_API_KEY") or os.getenv("GOOGLE_API_KEY")
    if key:
        print("Key loaded from .env / environment variable")
    return key

gemini_api_key = get_gemini_key()
if not gemini_api_key:
    raise ValueError("Gemini key not found. Colab: add a secret named GEMINI_API_KEY ...")

GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
gemini_client = AsyncOpenAI(api_key = gemini_api_key, base_url = GEMINI_BASE_URL)

set_tracing_disabled(True)
print("Gemini client successfully configured.")
print(gemini_api_key[:5])


Key loaded from Colab Secrets (GEMINI_API_KEY)
Gemini client successfully configured.
AQ.Ab


In [ ]:
GEMINI_MODEL = "gemini-3.8-flash"           # smart + fast, free tier (default)
GEMINI_LITE_MODEL = "gemini-3.5-flash-lite" # fastest, for simple tasks

# OPTIONAL: list the Gemini models your key can use
models = await gemini_client.models.list()
for m in models.data:
    if "gemini" in m.id:
        print(m.id.replace("models/", ""))


gemini-2.5-flash
gemini-2.5-pro
gemini-2.5-flash-preview-tts
gemini-2.5-pro-preview-tts
gemini-flash-latest
gemini-flash-lite-latest
gemini-pro-latest
gemini-2.5-flash-lite
gemini-2.5-flash-image
gemini-3-flash-preview
gemini-3.1-pro-preview
gemini-3.1-pro-preview-customtools
gemini-3.1-flash-lite-preview
gemini-3.1-flash-lite
gemini-3-pro-image-preview
gemini-3-pro-image
gemini-3.1-flash-image-preview
gemini-3.1-flash-image
gemini-3.1-flash-lite-image
gemini-nano-banana-2.1
gemini-3.5-flash
gemini-3.5-flash-lite
gemini-omni-flash-preview
gemini-omni-1.1-flash
gemini-3.5-transcribe
gemini-3.6-flash
gemini-3.7-flash
gemini-3.8-flash
gemini-4-argon
gemini-3.1-flash-tts-preview
gemini-3.8-flash-tts
gemini-3.8-flash-lite-tts
gemini-robotics-er-2-preview
gemini-2.5-computer-use-preview-10-2025
gemini-embedding-001
gemini-embedding-2-preview
gemini-embedding-2
gemini-3.5-transcribe-live
gemini-2.5-flash-native-audio-latest
gemini-2.5-flash-native-audio-preview-09-2025
gemini-2.5-flash-native

In [ ]:

fact_checker_instructions = """
Context:
You are a fact-checker who verifies the accuracy of statements.

Instructions:
When given a statement, carefully analyze its factual accuracy using your knowledge.

Input:
You will receive a statement that requires fact-checking.

Output:
Respond with:
1. A verdict prefix: either "✅ TRUE:" or "❌ FALSE:"
2. A brief, one-sentence explanation justifying your conclusion
3. if the input is not a fact checking question and its a general question donot use any verdict prefix, give just a plane response
"""

In [ ]:
from agents import Agent, OpenAIChatCompletionsModel

gemini_model = OpenAIChatCompletionsModel(model = GEMINI_LITE_MODEL,          # which Gemini brain
                                          openai_client = gemini_client) # connection to Google

fact_checker_agent = Agent(name = "Fact Checker",
                           instructions = fact_checker_instructions,
                           model = gemini_model)
print(f"Agent '{fact_checker_agent.name}' created successfully!")


Agent 'Fact Checker' created successfully!


In [ ]:
def print_markdown(text):
    """Displays text as Markdown in Jupyter."""
    display(Markdown(text))

In [ ]:
from agents import Runner

statement = "The Great Wall of China is visible from space with the naked eye."
response = await Runner.run(starting_agent = fact_checker_agent, input = statement)
print_markdown(response.final_output)


❌ FALSE: The Great Wall of China is generally not visible to the naked eye from low Earth orbit without magnification, as its width is too narrow and it blends in with the natural surrounding terrain.

In [ ]:
check_recall_statement = "What did we discuss in the last message?"
response = await Runner.run(starting_agent = fact_checker_agent, input = check_recall_statement)
print_markdown(response.final_output)


This is our very first message in this conversation, so we haven't discussed anything yet.

In [ ]:
def show_usage(result):
    usage = result.context_wrapper.usage
    print(f"Requests made   : {usage.requests}")
    print(f"Input tokens    : {usage.input_tokens}")
    print(f"Output tokens   : {usage.output_tokens}")
    print(f"Total tokens    : {usage.total_tokens}")

show_usage(response)


Requests made   : 1
Input tokens    : 134
Output tokens   : 19
Total tokens    : 153


In [ ]:
class TokenTracker:
    def __init__(self):
        self.total_requests = 0
        self.total_input_tokens = 0
        self.total_output_tokens = 0
        self.total_tokens = 0

    def add_usage(self, result):
        usage = result.context_wrapper.usage
        self.total_requests += usage.requests
        self.total_input_tokens += usage.input_tokens
        self.total_output_tokens += usage.output_tokens
        self.total_tokens += usage.total_tokens

    def display_summary(self):
        print("=== Cumulative Token Usage ===")
        print(f"Total Requests      : {self.total_requests}")
        print(f"Total Input Tokens  : {self.total_input_tokens}")
        print(f"Total Output Tokens : {self.total_output_tokens}")
        print(f"Grand Total Tokens  : {self.total_tokens}")

# Let's recreate the steps to calculate the total tokens consumed up to this point
tracker = TokenTracker()

# 1. Run for first statement
response_1 = await Runner.run(starting_agent = fact_checker_agent, input = statement)
tracker.add_usage(response_1)

# 2. Run for the recall statement
response_2 = await Runner.run(starting_agent = fact_checker_agent, input = check_recall_statement)
tracker.add_usage(response_2)

# Show accumulated usage
tracker.display_summary()

=== Cumulative Token Usage ===
Total Requests      : 2
Total Input Tokens  : 269
Total Output Tokens : 70
Grand Total Tokens  : 339


In [ ]:
statement = "The tallest mountain in the world is Mount Everst"
response = await Runner.run(starting_agent = fact_checker_agent, input = statement)
print_markdown(response.final_output)


✅ TRUE: Mount Everest is universally recognized as the tallest mountain in the world, with a peak at 8,848.86 meters (29,031.7 feet) above sea level.

In [ ]:
show_usage(response)

Requests made   : 1
Input tokens    : 135
Output tokens   : 45
Total tokens    : 180
